In [ ]:
import numpy as np
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from pathlib import Path
import string


class Config:
    IMG_WIDTH = 700
    IMG_HEIGHT = 200
    CHARACTERS = string.ascii_uppercase + string.digits  # 36 chars: A-Z, 0-9
    MAX_LENGTH = 5
    BATCH_SIZE = 16
    EPOCHS = 50
    LEARNING_RATE = 0.001
    TRAIN_DIR = "train-dataset/"
    TEST_DIR = "test-dataset/"
    MODEL_PATH = "captcha_model.h5"

config = Config()


def preprocess_image(img_path):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Image not found or unreadable: {img_path}")
    img = cv2.resize(img, (config.IMG_WIDTH, config.IMG_HEIGHT))
    img = img.astype(np.float32) / 255.0
    return np.expand_dims(img, axis=-1)


#Encodes a 6-character label into 6 class indices
def encode_label(text):
    text = text.upper()
    if len(text) != config.MAX_LENGTH:
        raise ValueError(f"Label must be exactly {config.MAX_LENGTH} characters. Got: '{text}'")
    return np.array([config.CHARACTERS.index(char) for char in text])


#Decodes 6 class indices back to a string.
def decode_label(encoded):
    return ''.join(config.CHARACTERS[idx] for idx in encoded)


class CaptchaDataGenerator(keras.utils.Sequence):
    def __init__(self, image_paths, batch_size=32, shuffle=True):
        self.image_paths = image_paths
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.image_paths))
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.image_paths) / self.batch_size))

    def __getitem__(self, idx):
        batch_indices = self.indices[
            idx * self.batch_size:(idx + 1) * self.batch_size
        ]
        batch_images = []
        batch_labels = []

        for i in batch_indices:
            img_path = self.image_paths[i]
            filename = Path(img_path).stem
            label = filename.split('_')[0]
            img = preprocess_image(img_path)
            batch_images.append(img)
            batch_labels.append(encode_label(label))

        batch_images = np.array(batch_images)
        batch_labels = np.array(batch_labels)

        label_dict = {}
        for i in range(config.MAX_LENGTH):
            label_dict[f"char_{i}"] = batch_labels[:, i]

        return batch_images, label_dict

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


def build_model():
    input_img = layers.Input(
        shape=(config.IMG_HEIGHT, config.IMG_WIDTH, 1),
        name='input_image'
    )

    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(input_img)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.2)(x)

    shape = x.shape
    x = layers.Reshape((shape[1] * shape[2], shape[3]))(x)

    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.Dropout(0.3)(x)

    outputs = []
    num_classes = len(config.CHARACTERS)
    for i in range(config.MAX_LENGTH):
        x_pos = layers.GlobalAveragePooling1D()(x)
        out = layers.Dense(num_classes, activation='softmax', name=f'char_{i}')(x_pos)
        outputs.append(out)

    model = keras.Model(inputs=input_img, outputs=outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config.LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'] * config.MAX_LENGTH 
    )
    return model


# TRAINING
def train_model():
    print("Loading training data")
    train_paths = list(Path(config.TRAIN_DIR).glob("*.png")) + list(Path(config.TRAIN_DIR).glob("*.jpg"))
    print(f"Found {len(train_paths)} training images")

    if len(train_paths) == 0:
        raise ValueError(f"No images found in {config.TRAIN_DIR}")

    train_gen = CaptchaDataGenerator(train_paths, batch_size=config.BATCH_SIZE)

    print("Building model")
    model = build_model()
    model.summary()

    callbacks = [
        keras.callbacks.ModelCheckpoint(config.MODEL_PATH, save_best_only=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-7, verbose=1),
        keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, verbose=1),
        keras.callbacks.TensorBoard(log_dir='./logs', histogram_freq=1)
    ]

    print("Starting training...")
    history = model.fit(train_gen, epochs=config.EPOCHS, callbacks=callbacks)
    plot_training_history(history)
    return model, history

def plot_training_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].set_title('Model Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(True)

    acc_keys = [f'char_{i}_accuracy' for i in range(config.MAX_LENGTH)]
    train_acc = np.mean([history.history[k] for k in acc_keys], axis=0)
    axes[1].plot(train_acc, label='Mean Train Accuracy')
    axes[1].set_title('Mean Character Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].legend(); axes[1].grid(True)

    plt.tight_layout()
    plt.savefig('training_history.png')
    plt.show()


# PREDICTION AND EVALUATION
def predict_captcha(model, img_path):
    img = preprocess_image(img_path)
    img = np.expand_dims(img, axis=0)
    predictions = model.predict(img, verbose=0)
    predicted_indices = [int(np.argmax(pred[0])) for pred in predictions]
    return decode_label(predicted_indices)

def evaluate_model(model, test_dir):
    test_paths = list(Path(test_dir).glob("*.png")) + list(Path(test_dir).glob("*.jpg"))
    print(f"\nEvaluating on {len(test_paths)} test images...")

    correct = 0
    char_correct = 0
    total_chars = 0

    for img_path in test_paths:
        filename = Path(img_path).stem
        true_label = filename.split('_')[0].upper()
        pred_label = predict_captcha(model, img_path)

        if pred_label == true_label:
            correct += 1

        for t, p in zip(true_label, pred_label):
            char_correct += (t == p)
            total_chars += 1

    full_acc = correct / len(test_paths) * 100
    char_acc = char_correct / total_chars * 100

    print(f"Results:")
    print(f"Full CAPTCHA Accuracy: {full_acc:.2f}%")
    print(f"Character-level Accuracy: {char_acc:.2f}%")
    print(f"Correct: {correct}/{len(test_paths)}")

    return {'full_accuracy': full_acc, 'char_accuracy': char_acc}

def visualize_predictions(model, test_dir, num_samples=10):
    test_paths = list(Path(test_dir).glob("*.png")) + list(Path(test_dir).glob("*.jpg"))
    if not test_paths:
        print("No test images found.")
        return

    sample_paths = np.random.choice(test_paths, min(num_samples, len(test_paths)), replace=False)
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes = axes.flatten()

    for idx, img_path in enumerate(sample_paths):
        img_display = cv2.imread(str(img_path))
        img_display = cv2.cvtColor(img_display, cv2.COLOR_BGR2RGB)

        true_label = Path(img_path).stem.split('_')[0].upper()
        pred_label = predict_captcha(model, img_path)

        axes[idx].imshow(img_display)
        axes[idx].axis('off')
        color = 'green' if pred_label == true_label else 'red'
        axes[idx].set_title(f'True: {true_label}\nPred: {pred_label}', color=color, fontsize=10)

    plt.tight_layout()
    plt.savefig('predictions_sample.png')
    plt.show()




if __name__ == "__main__":
    print('----------------')
    print("CAPTCHA Recognition Project")


    model, history = train_model()
    print("\nTraining complete!")


    print("Loading model for evaluation")
    model = keras.models.load_model(config.MODEL_PATH)
    results = evaluate_model(model, config.TEST_DIR)
    visualize_predictions(model, config.TEST_DIR, num_samples=10)